<a href="https://colab.research.google.com/github/topboyasian-stack/gta6-autoclipper/blob/main/GTA6_AutoClipper_GrowthReady.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GTA AutoClipper — Growth-Ready Kick workflow with captions and editorial packaging

Reuse this notebook for different Kick creators. Enter a creator, an optional game name, and a public clip/VOD URL. The notebook downloads directly where supported, transcribes speech, ranks possible moments, and renders vertical Shorts.

**Automatic editing and growth preparation:**
- Starts with a brief replay teaser from a high-energy moment (no logo intro), then plays the selected clip.
- Adds a short on-screen hook based on nearby recognised speech, or a neutral category-safe fallback.
- Burns in progressive word-highlight captions when speech is detected; silence is never filled with invented dialogue.
- Classifies each candidate into a tentative format: funny, clutch, unexpected, smart play, or general gameplay highlight.
- Creates title options, a description, relevant hashtags, a pinned-question suggestion, and TikTok/Instagram copy.
- Adds a simple performance-tracking CSV template for 24-hour, 7-day, and 28-day reviews.

**Important limits:** Candidate ranking combines audio/transcript signals with a lightweight visual-motion proxy from one sampled, downscaled frame per second. It does not understand what is happening on screen or identify kills, clutches, jokes, or story context. Review every clip, hook, caption, title, and category. This tool does not publish, comment, cross-post, or certify monetization eligibility. Captions, cropping, and a teaser alone do not guarantee that third-party footage becomes sufficiently original for YouTube monetization; use meaningful original commentary/editorial work and obtain the rights required for each use.

Defaults: 40-second target and 3 candidates. For multiple distinct candidates, use a longer VOD. `test` mode is for private technical testing only and still requires authorisation to download/process the source. `publish` mode additionally asks you to confirm permission/licence covering editing and YouTube reposting. Do not bypass subscriber restrictions, access controls, or platform download restrictions.


In [ ]:
# 1) Enter the source and settings for THIS creator/session
import re
from urllib.parse import urlparse

CREATOR_NAME = input("Creator/channel name: ").strip()
GAME_NAME = input("Game name (optional, for titles/hashtags): ").strip()
SOURCE_URL = input("Paste the public Kick clip or VOD URL: ").strip()

SESSION_MODE = (input("Session mode [test/publish] (default: test): ").strip().lower() or "test")
assert SESSION_MODE in {"test", "publish"}, "Choose test or publish."

raw_target = input("Target clip length in seconds [40]: ").strip()
TARGET_SECONDS = int(raw_target) if raw_target else 40
raw_candidates = input("Maximum candidates [3]: ").strip()
MAX_CANDIDATES = int(raw_candidates) if raw_candidates else 3

SOURCE_IS_COMPLETE_AND_PUBLIC_OR_OWN_FOOTAGE = input(
    "Has the stream ended and is this source public (not subscriber-only), or is it your own footage? Type YES: "
).strip().lower() in {"yes", "y"}
RIGHTS_COVER_DOWNLOAD_AND_ANALYSIS = input(
    "Are you authorised to download and process this source for this session (ownership, permission, or applicable licence)? Type YES: "
).strip().lower() in {"yes", "y"}

RIGHTS_COVER_EDITING_AND_YOUTUBE_REPOSTING = False
if SESSION_MODE == "publish":
    RIGHTS_COVER_EDITING_AND_YOUTUBE_REPOSTING = input(
        "Do you have permission/licence covering BOTH editing and YouTube reposting? Type YES: "
    ).strip().lower() in {"yes", "y"}

RIGHTS_BASIS = input("Describe the permission/licence basis for this source: ").strip()

parsed_url = urlparse(SOURCE_URL)
host = (parsed_url.hostname or "").lower()
assert 10 <= TARGET_SECONDS <= 90, "Target length must be 10–90 seconds."
assert 1 <= MAX_CANDIDATES <= 6, "Candidate count must be 1–6."
assert CREATOR_NAME, "Creator/channel name is required."
assert parsed_url.scheme in {"http", "https"} and (host == "kick.com" or host.endswith(".kick.com")), "Enter a public Kick clip or VOD URL."
assert SOURCE_IS_COMPLETE_AND_PUBLIC_OR_OWN_FOOTAGE, "Stopped: use only a completed public source or footage you own."
assert RIGHTS_COVER_DOWNLOAD_AND_ANALYSIS, "Stopped: you must be authorised to download and process this source."
assert RIGHTS_BASIS, "Describe the actual permission/licence basis before processing."
if SESSION_MODE == "publish":
    assert RIGHTS_COVER_EDITING_AND_YOUTUBE_REPOSTING, "Stopped: publishing requires rights covering editing AND YouTube reposting."

CREATOR_SLUG = re.sub(r"[^a-z0-9]+", "_", CREATOR_NAME.lower()).strip("_") or "creator"
print("\nSettings validated.")
print(f"Creator: {CREATOR_NAME} | Mode: {SESSION_MODE} | Target: {TARGET_SECONDS}s | Max candidates: {MAX_CANDIDATES}")
if SESSION_MODE == "test":
    print("TEST MODE: keep the generated files private; this mode does not grant reposting rights.")
print("The downloader will use the Kick URL directly; manual upload is only a fallback if direct extraction fails.")


In [ ]:
# 2) Install the downloader, then start the URL download in the background.
# This cell returns as soon as the background job starts, so setup can happen in parallel.
!pip -q install -U yt-dlp

import os, glob, threading, time, traceback
from pathlib import Path
from yt_dlp import YoutubeDL

WORK_DIR = Path("/content/gta6_autoclipper")
WORK_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_TEMPLATE = str(WORK_DIR / "kick_source.%(ext)s")
DOWNLOAD_STATE = {"done": False, "error": None, "path": None, "info": None}

# Remove leftovers from a previous attempt so the notebook cannot mistake an old file for this download.
for old_file in WORK_DIR.glob("kick_source.*"):
    try:
        old_file.unlink()
    except OSError:
        pass

def _download_kick_source():
    try:
        options = {
            "format": "best[ext=mp4]/best",
            "outtmpl": DOWNLOAD_TEMPLATE,
            "merge_output_format": "mp4",
            "noplaylist": True,
            "retries": 5,
            "fragment_retries": 5,
            "socket_timeout": 30,
            "continuedl": True,
            "quiet": False,
            "no_warnings": False,
        }
        with YoutubeDL(options) as ydl:
            info = ydl.extract_info(SOURCE_URL, download=True)
            DOWNLOAD_STATE["info"] = info
        candidates = [p for p in WORK_DIR.glob("kick_source.*") if p.is_file() and p.stat().st_size > 0]
        # Prefer the merged MP4 when yt-dlp has produced one.
        candidates.sort(key=lambda p: (p.suffix.lower() != ".mp4", -p.stat().st_size))
        if not candidates:
            raise FileNotFoundError("yt-dlp finished but no downloaded media file was found.")
        DOWNLOAD_STATE["path"] = str(candidates[0])
    except Exception as exc:
        DOWNLOAD_STATE["error"] = f"{type(exc).__name__}: {exc}"
        DOWNLOAD_STATE["traceback"] = traceback.format_exc()
    finally:
        DOWNLOAD_STATE["done"] = True

DOWNLOAD_THREAD = threading.Thread(target=_download_kick_source, name="kick-url-download", daemon=True)
DOWNLOAD_THREAD.start()
print("Kick download started in the background.")
print("Continue to the next cell now — FFmpeg and the speech-analysis tools will be prepared while the file downloads.")

In [ ]:
# 3) Prepare media, caption rendering, and speech-analysis tools while Kick downloads.
!apt-get -qq update
!apt-get -qq install -y ffmpeg fonts-dejavu-core
!pip -q install -U faster-whisper soundfile "numpy<2.5"

import shutil, subprocess, json, math, re
import numpy as np
import soundfile as sf

assert shutil.which("ffmpeg"), "FFmpeg installation did not complete."
assert shutil.which("ffprobe"), "ffprobe installation did not complete."
_ffmpeg_filters = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True).stdout
HAS_SUBTITLES_FILTER = "subtitles" in _ffmpeg_filters
HAS_DRAWTEXT_FILTER = "drawtext" in _ffmpeg_filters
print("FFmpeg subtitles/libass filter:", "available" if HAS_SUBTITLES_FILTER else "not available")
print("FFmpeg drawtext hook overlay:", "available" if HAS_DRAWTEXT_FILTER else "not available")
print("Processing tools are ready. The source download may still be running; the next cell checks it.")


In [ ]:
# 4) Wait for the URL download to finish and inspect the downloaded source.
from IPython.display import display, Markdown

DOWNLOAD_THREAD.join()
if DOWNLOAD_STATE["error"]:
    print("Direct Kick download failed:")
    print(DOWNLOAD_STATE["error"])
    print("\nFallback: download the clip/VOD using Kick's own available download option or another method you are authorised to use, then select it below.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No fallback file was selected.")
    uploaded_name = next(iter(uploaded))
    fallback_path = WORK_DIR / Path(uploaded_name).name
    fallback_path.write_bytes(uploaded[uploaded_name])
    VIDEO_PATH = str(fallback_path)
else:
    VIDEO_PATH = DOWNLOAD_STATE["path"]

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration:stream=codec_type,width,height", "-of", "json", VIDEO_PATH],
    capture_output=True, text=True, check=True
)
probe_data = json.loads(probe.stdout)
DURATION = float(probe_data.get("format", {}).get("duration", 0))
video_stream = next((s for s in probe_data.get("streams", []) if s.get("codec_type") == "video"), {})
WIDTH, HEIGHT = int(video_stream.get("width", 0)), int(video_stream.get("height", 0))
HAS_AUDIO = any(s.get("codec_type") == "audio" for s in probe_data.get("streams", []))
if DURATION < 10:
    raise RuntimeError(f"The source is only {DURATION:.1f}s long; at least 10 seconds are needed.")

print(f"Downloaded: {Path(VIDEO_PATH).name}")
print(f"Duration: {DURATION:.1f}s | Resolution: {WIDTH}×{HEIGHT} | Audio: {'yes' if HAS_AUDIO else 'no'}")
if DURATION < TARGET_SECONDS:
    print(f"Note: the source is shorter than the {TARGET_SECONDS}s target. Candidate length will be capped at {DURATION:.1f}s.")
if DURATION < TARGET_SECONDS * 1.6:
    print("Note: this source may not contain enough footage for three meaningfully different candidates.")


In [ ]:
# 5) Extract audio and transcribe it with word-level timestamps for captions.
# Pass a NumPy audio array to Whisper rather than a file path. This avoids a
# PyAV file-decoding compatibility issue seen in some Colab runtimes.
AUDIO_PATH = str(WORK_DIR / "source_audio_16k.wav")
subprocess.run(
    ["ffmpeg", "-y", "-i", VIDEO_PATH, "-vn", "-ac", "1", "-ar", "16000", AUDIO_PATH],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
)

def transcribe_source(path):
    try:
        from faster_whisper import WhisperModel
        samples, sample_rate = sf.read(path, dtype="float32")
        if samples.ndim > 1:
            samples = samples.mean(axis=1)
        if sample_rate != 16000:
            raise ValueError(f"Expected 16 kHz audio, got {sample_rate} Hz")

        # Multilingual tiny model detects the language automatically.
        model = WhisperModel("tiny", device="cpu", compute_type="int8")
        segments, info = model.transcribe(
            samples,
            beam_size=1,
            vad_filter=True,
            word_timestamps=True,
            condition_on_previous_text=False
        )

        rows = []
        for seg in segments:
            word_rows = []
            for word in (getattr(seg, "words", None) or []):
                word_text = (word.word or "").strip()
                if word_text:
                    word_rows.append({
                        "start": float(word.start),
                        "end": float(word.end),
                        "text": word_text
                    })
            rows.append({
                "start": float(seg.start),
                "end": float(seg.end),
                "text": seg.text.strip(),
                "words": word_rows
            })
        total_words = sum(len(row["words"]) for row in rows)
        print(f"Transcription language: {info.language} | speech segments: {len(rows)} | timed words: {total_words}")
        return rows
    except Exception as exc:
        print(f"Speech transcription unavailable ({type(exc).__name__}: {exc}). Captions will be omitted rather than fabricated.")
        return []

TRANSCRIPT = transcribe_source(AUDIO_PATH)
print("Audio analysis and timestamped transcription are ready.")

In [ ]:
# 6) Rank candidate windows using speech, energetic audio, and sampled visual motion.
# Visual motion is only a proxy: it can surface fast gameplay, but cannot identify a kill,
# clutch, funny moment, or the meaning of a scene. Review every selected clip.

audio, sample_rate = sf.read(AUDIO_PATH, dtype="float32")
if audio.ndim > 1:
    audio = audio.mean(axis=1)

window_seconds = min(float(TARGET_SECONDS), DURATION)
hop = 5.0
last_start = max(0.0, DURATION - window_seconds)
starts = np.arange(0.0, last_start + 0.001, hop).tolist()
if not starts:
    starts = [0.0]
if starts[-1] < last_start - 0.5:
    starts.append(last_start)

# Build a 0.5-second RMS loudness curve.
frame_samples = max(1, int(sample_rate * 0.5))
rms_values = []
for i in range(0, len(audio), frame_samples):
    chunk = audio[i:i+frame_samples]
    rms_values.append(float(np.sqrt(np.mean(chunk * chunk))) if len(chunk) else 0.0)
rms_values = np.asarray(rms_values, dtype=np.float32)

# Sample one small grayscale frame per second with FFmpeg. This avoids installing
# OpenCV or downloading a separate vision model, and keeps memory use low.
VISUAL_WIDTH, VISUAL_HEIGHT = 160, 90
VISUAL_FRAME_BYTES = VISUAL_WIDTH * VISUAL_HEIGHT
visual_deltas = []
visual_sampling_error = None

def _read_exact(stream, size):
    chunks = []
    remaining = size
    while remaining:
        chunk = stream.read(remaining)
        if not chunk:
            break
        chunks.append(chunk)
        remaining -= len(chunk)
    data = b"".join(chunks)
    return data if len(data) == size else None

ffmpeg_visual_command = [
    "ffmpeg", "-hide_banner", "-loglevel", "error",
    "-i", str(VIDEO_PATH),
    "-vf",
    "fps=1,scale=160:90:force_original_aspect_ratio=decrease,"
    "pad=160:90:(ow-iw)/2:(oh-ih)/2,format=gray",
    "-an", "-sn", "-dn",
    "-f", "rawvideo", "-pix_fmt", "gray", "pipe:1"
]

try:
    visual_proc = subprocess.Popen(
        ffmpeg_visual_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    previous_frame = None
    frame_index = 0
    while True:
        raw_frame = _read_exact(visual_proc.stdout, VISUAL_FRAME_BYTES)
        if raw_frame is None:
            break
        current_frame = np.frombuffer(raw_frame, dtype=np.uint8).astype(np.int16)
        if previous_frame is not None:
            # Mean absolute pixel change, normalised to 0–1.
            delta = float(np.mean(np.abs(current_frame - previous_frame)) / 255.0)
            visual_deltas.append(delta)
        previous_frame = current_frame
        frame_index += 1

    visual_stderr = visual_proc.stderr.read().decode("utf-8", errors="replace").strip()
    visual_return_code = visual_proc.wait()
    if visual_return_code != 0:
        raise RuntimeError(visual_stderr or f"FFmpeg exited with code {visual_return_code}")
    if len(visual_deltas) < 2:
        raise RuntimeError("Not enough sampled frames to calculate visual motion.")
except Exception as exc:
    visual_sampling_error = f"{type(exc).__name__}: {exc}"
    try:
        if "visual_proc" in locals() and visual_proc.poll() is None:
            visual_proc.kill()
            visual_proc.wait()
    except Exception:
        pass
    visual_deltas = []

visual_deltas = np.asarray(visual_deltas, dtype=np.float32)
VISUAL_MOTION_AVAILABLE = len(visual_deltas) >= 2
if VISUAL_MOTION_AVAILABLE:
    visual_reference = max(float(np.percentile(visual_deltas, 90)), 1e-6)
    print(
        f"Visual motion sampling ready: {len(visual_deltas)} one-second frame changes "
        f"({VISUAL_WIDTH}x{VISUAL_HEIGHT} grayscale)."
    )
else:
    print("Visual motion sampling unavailable; ranking will use the audio-only fallback.")
    if visual_sampling_error:
        print("Visual sampling note:", visual_sampling_error)

HIGHLIGHT_TERMS = {
    "insane", "crazy", "no way", "oh my god", "let's go", "lets go", "what", "wow",
    "clutch", "squad", "wipe", "killed", "kill", "headshot", "win", "winner",
    "bro", "enemy", "one shot", "last one", "nice", "holy", "damn"
}

def window_stats(start, end):
    a = max(0, int(start / 0.5))
    b = min(len(rms_values), max(a + 1, int(math.ceil(end / 0.5))))
    vals = rms_values[a:b]
    mean_energy = float(np.mean(vals)) if len(vals) else 0.0
    peak_energy = float(np.percentile(vals, 90)) if len(vals) else 0.0
    variability = float(np.std(vals)) if len(vals) else 0.0

    speech_rows = [x for x in TRANSCRIPT if x["end"] > start and x["start"] < end]
    spoken_words = sum(len(x["text"].split()) for x in speech_rows)
    text = " ".join(x["text"].lower() for x in speech_rows)
    phrase_hits = sum(1 for term in HIGHLIGHT_TERMS if term in text)

    # Normalise audio features against this source.
    audio_denom = float(np.percentile(rms_values, 95)) + 1e-6 if len(rms_values) else 1.0
    energy_score = min(1.0, (0.55 * mean_energy + 0.45 * peak_energy) / audio_denom)
    speech_score = min(1.0, spoken_words / max(1, window_seconds * 2.2))
    phrase_score = min(1.0, phrase_hits / 3.0)
    dynamics_score = (
        min(1.0, variability / (float(np.percentile(rms_values, 90)) + 1e-6))
        if len(rms_values) else 0.0
    )

    audio_score = (
        0.42 * energy_score
        + 0.28 * speech_score
        + 0.20 * phrase_score
        + 0.10 * dynamics_score
    )

    visual_score = None
    if VISUAL_MOTION_AVAILABLE:
        # Each delta represents the visual change over one sampled second.
        va = max(0, int(math.floor(start)))
        vb = min(len(visual_deltas), max(va + 1, int(math.ceil(end))))
        motion_values = visual_deltas[va:vb]
        if len(motion_values):
            motion_mean = float(np.mean(motion_values))
            motion_burst = float(np.percentile(motion_values, 90))
            visual_score = min(
                1.0,
                (0.60 * motion_mean + 0.40 * motion_burst) / visual_reference
            )
        else:
            visual_score = 0.0

    # Visual motion contributes 24%; audio/transcript remains the main signal.
    # If frame sampling fails, retain the original audio-only ranking weights.
    total = 0.76 * audio_score + 0.24 * visual_score if visual_score is not None else audio_score

    return {
        "start": float(start),
        "end": float(end),
        "score": float(total),
        "audio_score": float(audio_score),
        "visual_score": None if visual_score is None else float(visual_score),
        "speech_words": int(spoken_words),
        "phrase_hits": int(phrase_hits),
        "transcript": text[:500]
    }

scored = [window_stats(s, min(DURATION, s + window_seconds)) for s in starts]
scored.sort(key=lambda x: x["score"], reverse=True)

selected = []
for item in scored:
    overlaps = False
    for chosen in selected:
        overlap = max(0.0, min(item["end"], chosen["end"]) - max(item["start"], chosen["start"]))
        shorter = min(item["end"] - item["start"], chosen["end"] - chosen["start"])
        if shorter > 0 and overlap / shorter > 0.35:
            overlaps = True
            break
    if not overlaps:
        selected.append(item)
    if len(selected) >= min(MAX_CANDIDATES, len(scored)):
        break

if not selected:
    selected = [window_stats(0.0, min(DURATION, window_seconds))]

print(f"Selected {len(selected)} candidate window(s).")
for i, item in enumerate(selected, 1):
    visual_label = (
        f"{item['visual_score']:.3f}" if item["visual_score"] is not None else "unavailable"
    )
    print(
        f"Candidate {i}: {item['start']:.1f}s–{item['end']:.1f}s"
        f" | combined {item['score']:.3f}"
        f" | audio {item['audio_score']:.3f}"
        f" | visual-motion {visual_label}"
        f" | speech words {item['speech_words']}"
        f" | highlight phrases {item['phrase_hits']}"
    )
    if item["transcript"]:
        print("  ", item["transcript"][:220])


In [ ]:
# 7) Turn the ranked moments into an editorial plan and publishing pack.
# These are grounded suggestions based on recognised speech and audio activity.
# They are NOT computer-vision judgements and should be reviewed before publishing.
import csv, json, re, textwrap
from pathlib import Path

CATEGORY_TERMS = {
    "Funny moments": ["haha", "hahaha", "lol", "lmao", "funny", "bruh", "oops", "my bad", "what was that"],
    "Clutch moments": ["clutch", "last one", "1v", "one versus", "headshot", "wipe", "winner", "we won", "i won", "survived", "revive", "knocked"],
    "Unexpected moments": ["no way", "what", "wow", "oh", "ah", "surprise", "unexpected", "air drop", "airdrop", "didn't expect", "did not expect", "where did"],
    "Smart plays": ["rotate", "rotation", "cover me", "flank", "strategy", "position", "angle", "bait", "wait", "push", "hold", "plan"],
}

CATEGORY_HOOKS = {
    "Funny moments": "THIS DID NOT GO TO PLAN",
    "Clutch moments": "WATCH THIS CLUTCH MOMENT",
    "Unexpected moments": "NOBODY SAW THIS COMING",
    "Smart plays": "WATCH THE PLAY UNFOLD",
    "Gameplay highlights": "WAIT FOR THE GAME-CHANGING MOMENT",
}

CATEGORY_QUESTIONS = {
    "Funny moments": "Was that skill, bad luck, or just hilarious?",
    "Clutch moments": "Would you have pulled that off?",
    "Unexpected moments": "Did you see that coming?",
    "Smart plays": "Would you have made the same move?",
    "Gameplay highlights": "What would you have done here?",
}

CATEGORY_TITLES = {
    "Funny moments": ["This did NOT go to plan 😂", "The reaction makes this even funnier 😭", "Gaming moments you can't script"],
    "Clutch moments": ["A clutch moment under pressure 🔥", "That timing was everything 😳", "Would you have made this play?"],
    "Unexpected moments": ["Nobody saw this moment coming 😳", "Wait for the unexpected moment…", "That reaction says it all 👀"],
    "Smart plays": ["A smart move under pressure 🎯", "The decision that changed the fight", "Would you make this play?"],
    "Gameplay highlights": ["A tense gaming moment 👀", "Watch how this moment unfolds", "What would you do here?"]
}


def classify_candidate(item):
    text = (item.get("transcript") or "").lower()
    scores = {}
    for category, terms in CATEGORY_TERMS.items():
        scores[category] = sum(1 for term in terms if term in text)
    best = max(scores, key=scores.get) if scores else "Gameplay highlights"
    top = scores.get(best, 0)
    ranked = sorted(scores.values(), reverse=True)
    # Do not force an unsupported category when there is little or ambiguous evidence.
    if top == 0 or (len(ranked) > 1 and ranked[0] == ranked[1]):
        return "Gameplay highlights", "low", scores
    confidence = "medium" if top == 1 else "higher"
    return best, confidence, scores


def choose_teaser_and_hook(item, teaser_seconds=1.6):
    clip_start, clip_end = float(item["start"]), float(item["end"])
    clip_duration = max(0.1, clip_end - clip_start)
    teaser_seconds = min(float(teaser_seconds), max(0.5, clip_duration / 4.0))
    frame_step = 0.5
    first = max(0, int(clip_start / frame_step))
    last = min(len(rms_values), int(math.ceil(clip_end / frame_step)))
    segment_values = rms_values[first:last]
    frame_count = max(1, int(round(teaser_seconds / frame_step)))
    if len(segment_values) >= frame_count:
        rolling = np.convolve(segment_values, np.ones(frame_count, dtype=np.float32) / frame_count, mode="valid")
        # Avoid a teaser beginning exactly at a clip boundary where possible.
        valid_lo = min(max(0, 1), max(0, len(rolling)-1))
        valid_hi = max(valid_lo + 1, len(rolling)-1)
        local_idx = int(valid_lo + np.argmax(rolling[valid_lo:valid_hi])) if valid_hi > valid_lo else 0
        teaser_local_start = local_idx * frame_step
    else:
        teaser_local_start = max(0.0, (clip_duration - teaser_seconds) / 2.0)
    teaser_local_start = min(max(0.0, teaser_local_start), max(0.0, clip_duration - teaser_seconds))
    teaser_abs_start = clip_start + teaser_local_start
    teaser_abs_end = teaser_abs_start + teaser_seconds

    nearby = [s for s in TRANSCRIPT if float(s.get("end", 0)) > teaser_abs_start - 0.8 and float(s.get("start", 0)) < teaser_abs_end + 1.2]
    def hook_segment_score(seg):
        overlap = max(0.0, min(float(seg.get("end", 0)), teaser_abs_end + 0.6) - max(float(seg.get("start", 0)), teaser_abs_start - 0.6))
        duration = max(0.0, float(seg.get("end", 0)) - float(seg.get("start", 0)))
        midpoint_distance = abs((float(seg.get("start", 0)) + float(seg.get("end", 0))) / 2 - (teaser_abs_start + teaser_abs_end) / 2)
        return overlap + min(0.5, duration * 0.2) - midpoint_distance * 0.03
    nearby.sort(key=hook_segment_score, reverse=True)
    spoken = (nearby[0].get("text", "") if nearby else "")
    spoken = re.sub(r"\s+", " ", spoken).strip(" .,!?:;-")
    spoken = re.sub(r"^(and|so|but|uh|um)\s+", "", spoken, flags=re.I)
    # Use recognisable source speech when available; otherwise use a safe category-level hook.
    hook = spoken.upper() if spoken else ""
    if len(hook) > 54:
        hook = hook[:51].rsplit(" ", 1)[0] + "…"
    return teaser_local_start, teaser_seconds, hook


def make_title_suggestions(category, hook, game_name):
    base = list(CATEGORY_TITLES.get(category, CATEGORY_TITLES["Gameplay highlights"]))
    if hook:
        clean = hook.title()
        base.insert(0, f'“{clean}” 👀')
    game_suffix = f" | {game_name}" if game_name else ""
    return [t + game_suffix for t in base[:3]]


def make_candidate_pack(item, number):
    category, confidence, category_scores = classify_candidate(item)
    teaser_start, teaser_duration, spoken_hook = choose_teaser_and_hook(item)
    hook = spoken_hook or CATEGORY_HOOKS[category]
    titles = make_title_suggestions(category, spoken_hook, GAME_NAME)
    game_tag = re.sub(r"[^A-Za-z0-9]", "", GAME_NAME.title()) if GAME_NAME else "Gaming"
    hashtags = ["#Gaming", "#GamingShorts", f"#{game_tag}"]
    description = (
        f"{category} from {CREATOR_NAME}. Watch the moment unfold.\n"
        f"Original creator: {CREATOR_NAME}.\n"
        f"Source: {SOURCE_URL}\n\n" + " ".join(hashtags)
    )
    question = CATEGORY_QUESTIONS[category]
    return {
        "candidate": number,
        "category": category,
        "category_confidence": confidence,
        "category_signal_counts": category_scores,
        "hook_text": hook,
        "teaser_start_in_clip_seconds": round(teaser_start, 2),
        "teaser_duration_seconds": round(teaser_duration, 2),
        "title_options": titles,
        "description": description,
        "hashtags": hashtags,
        "pinned_question": question,
        "tiktok_caption": f"{titles[0]}\n{question}\n" + " ".join(hashtags),
        "instagram_caption": f"{titles[0]}\n{description}",
        "editorial_review_note": "Check the visual context, captions, category and title. Audio/transcript heuristics do not understand the gameplay visuals.",
        "voiceover_guidance": "Optional: add one genuinely original sentence explaining the stakes or why the decision matters. Do not merely repeat the streamer's words. The notebook does not invent or synthesize visual commentary."
    }

EDITORIAL_PACK = [make_candidate_pack(item, i) for i, item in enumerate(selected, 1)]
print("Editorial plans prepared:")
for pack in EDITORIAL_PACK:
    print(f"Candidate {pack['candidate']}: {pack['category']} ({pack['category_confidence']} confidence)")
    print(f"  First-2-second hook: {pack['hook_text']}")
    print(f"  Title suggestion: {pack['title_options'][0]}")
    print(f"  Pinned question: {pack['pinned_question']}")


In [ ]:
# 8) Render vertical clips with a 1.6-second action teaser, hook text, and progressive captions.
# Each exported video starts with a short replay of a high-energy point, then the main clip.
# The teaser is included inside the target duration (the main segment is shortened accordingly).
OUTPUT_DIR = WORK_DIR / "candidates"
OUTPUT_DIR.mkdir(exist_ok=True)
for old in list(OUTPUT_DIR.glob("candidate_*.mp4")) + list(OUTPUT_DIR.glob("candidate_*.ass")) + list(OUTPUT_DIR.glob("candidate_*.srt")) + list(OUTPUT_DIR.glob("candidate_*_hook.txt")) + list(OUTPUT_DIR.glob("candidate_*_editorial.txt")):
    old.unlink()


def _ass_timestamp(seconds):
    cs = max(0, int(round(float(seconds) * 100)))
    hours, cs = divmod(cs, 360000)
    minutes, cs = divmod(cs, 6000)
    secs, centis = divmod(cs, 100)
    return f"{hours}:{minutes:02d}:{secs:02d}.{centis:02d}"


def _srt_timestamp(seconds):
    ms = max(0, int(round(float(seconds) * 1000)))
    hours, ms = divmod(ms, 3600000)
    minutes, ms = divmod(ms, 60000)
    secs, millis = divmod(ms, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


def _safe_caption_word(value):
    value = str(value).replace("\\", "").replace("{", "").replace("}", "")
    return re.sub(r"\s+", " ", value).strip()


def _all_timed_words():
    timed = []
    for segment in TRANSCRIPT:
        words = segment.get("words") or []
        if words:
            source_words = words
        else:
            tokens = re.findall(r"\S+", segment.get("text", ""))
            span = max(0.05, float(segment["end"]) - float(segment["start"]))
            source_words = []
            for token_index, token in enumerate(tokens):
                a = float(segment["start"]) + span * token_index / max(1, len(tokens))
                b = float(segment["start"]) + span * (token_index + 1) / max(1, len(tokens))
                source_words.append({"start": a, "end": b, "text": token})
        for word in source_words:
            token = _safe_caption_word(word.get("text", ""))
            if token:
                timed.append({"start": float(word["start"]), "end": float(word["end"]), "text": token})
    return sorted(timed, key=lambda x: (x["start"], x["end"]))

ALL_TIMED_WORDS = _all_timed_words()


def map_words_to_teaser_timeline(clip_start, clip_end, teaser_local_start, teaser_duration):
    teaser_abs_start = clip_start + teaser_local_start
    teaser_abs_end = teaser_abs_start + teaser_duration
    main_abs_end = clip_end - teaser_duration
    mapped = []
    for word in ALL_TIMED_WORDS:
        a, b = word["start"], word["end"]
        # First occurrence: replay teaser at output time zero.
        if b > teaser_abs_start and a < teaser_abs_end:
            mapped.append({
                "start": max(a, teaser_abs_start) - teaser_abs_start,
                "end": min(b, teaser_abs_end) - teaser_abs_start,
                "text": word["text"]
            })
        # Second occurrence: main clip follows the teaser, with its final teaser-length tail trimmed.
        if b > clip_start and a < main_abs_end:
            mapped.append({
                "start": teaser_duration + max(a, clip_start) - clip_start,
                "end": teaser_duration + min(b, main_abs_end) - clip_start,
                "text": word["text"]
            })
    mapped = [w for w in mapped if w["end"] > w["start"]]
    mapped.sort(key=lambda x: (x["start"], x["end"]))
    return mapped


def group_caption_words(words, max_words=5, max_chars=30):
    groups, current = [], []
    for word in words:
        candidate = current + [word]
        candidate_text = " ".join(x["text"] for x in candidate)
        punctuation_break = bool(current and re.search(r"[.!?]$", current[-1]["text"]))
        if current and (len(candidate) > max_words or len(candidate_text) > max_chars or punctuation_break):
            groups.append(current)
            current = [word]
        else:
            current = candidate
    if current:
        groups.append(current)
    return groups


def _wrap_ass_prefix(group, last_index, line_limit=22):
    lines, line_words, line_len = [], [], 0
    for idx, word in enumerate(group[:last_index + 1]):
        token = word["text"]
        extra = len(token) + (1 if line_words else 0)
        if line_words and line_len + extra > line_limit and len(lines) < 1:
            lines.append(line_words)
            line_words, line_len = [], 0
        line_words.append((idx, token))
        line_len += len(token) + (1 if len(line_words) > 1 else 0)
    if line_words:
        lines.append(line_words)
    rendered_lines = []
    for line in lines[:2]:
        rendered = []
        for idx, token in line:
            if idx == last_index:
                rendered.append(r"{\c&H0000FFFF&}" + token + r"{\c&H00FFFFFF&}")
            else:
                rendered.append(token)
        rendered_lines.append(" ".join(rendered))
    return r"\N".join(rendered_lines)


def build_caption_files(mapped_words, output_duration, ass_path, srt_path):
    if not mapped_words:
        ass_path.write_text("", encoding="utf-8")
        srt_path.write_text("", encoding="utf-8")
        return 0
    groups = group_caption_words(mapped_words)
    ass_lines = [
        "[Script Info]", "ScriptType: v4.00+", "PlayResX: 1080", "PlayResY: 1920",
        "WrapStyle: 2", "ScaledBorderAndShadow: yes", "", "[V4+ Styles]",
        "Format: Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,Alignment,MarginL,MarginR,MarginV,Encoding",
        "Style: Default,DejaVu Sans,62,&H00FFFFFF,&H00FFFFFF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,5,2,2,72,72,190,1",
        "", "[Events]", "Format: Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text"
    ]
    srt_lines, srt_index = [], 1
    for group in groups:
        phrase_start = max(0.0, group[0]["start"])
        phrase_end = min(output_duration, group[-1]["end"] + 0.32)
        phrase_text = " ".join(x["text"] for x in group)
        if phrase_end > phrase_start:
            srt_lines.extend([str(srt_index), f"{_srt_timestamp(phrase_start)} --> {_srt_timestamp(phrase_end)}", phrase_text, ""])
            srt_index += 1
        for word_index, word in enumerate(group):
            event_start = max(0.0, word["start"])
            if word_index + 1 < len(group):
                event_end = max(event_start + 0.08, group[word_index + 1]["start"])
            else:
                event_end = max(event_start + 0.12, word["end"] + 0.32)
            event_end = min(max(event_end, event_start + 0.08), output_duration)
            if event_end <= event_start:
                continue
            ass_lines.append(f"Dialogue: 0,{_ass_timestamp(event_start)},{_ass_timestamp(event_end)},Default,,0,0,0,,{_wrap_ass_prefix(group, word_index)}")
    ass_path.write_text("\n".join(ass_lines) + "\n", encoding="utf-8")
    srt_path.write_text("\n".join(srt_lines), encoding="utf-8")
    return len(mapped_words)


def _escape_filter_path(path):
    return Path(path).resolve().as_posix().replace(":", r"\:").replace("'", r"\'")

FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"

for idx, item in enumerate(selected, 1):
    pack = EDITORIAL_PACK[idx - 1]
    out_path = OUTPUT_DIR / f"candidate_{idx:02d}.mp4"
    ass_path = OUTPUT_DIR / f"candidate_{idx:02d}.ass"
    srt_path = OUTPUT_DIR / f"candidate_{idx:02d}.srt"
    hook_path = OUTPUT_DIR / f"candidate_{idx:02d}_hook.txt"
    editorial_path = OUTPUT_DIR / f"candidate_{idx:02d}_editorial.txt"

    start, end = float(item["start"]), float(item["end"])
    duration = end - start
    teaser_local_start = float(pack["teaser_start_in_clip_seconds"])
    teaser_duration = min(float(pack["teaser_duration_seconds"]), max(0.4, duration - 1.0))
    main_duration = duration - teaser_duration
    mapped_words = map_words_to_teaser_timeline(start, end, teaser_local_start, teaser_duration)
    caption_word_count = build_caption_files(mapped_words, duration, ass_path, srt_path)

    # Use at most two short lines for a clean top-safe-area hook.
    hook_display = "\n".join(textwrap.wrap(pack["hook_text"], width=24)[:2])
    hook_path.write_text(hook_display, encoding="utf-8")
    editorial_path.write_text(
        "\n".join([
            f"Content category: {pack['category']} ({pack['category_confidence']} confidence)",
            f"Hook overlay: {pack['hook_text']}",
            f"Teaser moment: {teaser_local_start:.2f}s into original candidate, {teaser_duration:.2f}s long",
            "Title options:", *[f"  - {t}" for t in pack["title_options"]],
            "", "Description:", pack["description"], "", "Hashtags: " + " ".join(pack["hashtags"]),
            "Pinned question suggestion: " + pack["pinned_question"],
            "TikTok caption:", pack["tiktok_caption"],
            "Instagram caption:", pack["instagram_caption"],
            "", "Editorial review:", pack["editorial_review_note"],
            pack["voiceover_guidance"], ""
        ]), encoding="utf-8"
    )

    base_graph = (
        "[0:v]split=2[bgsrc][fgsrc];"
        "[bgsrc]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,gblur=sigma=24[bg];"
        "[fgsrc]scale=1080:1920:force_original_aspect_ratio=decrease[fg];"
        "[bg][fg]overlay=(W-w)/2:(H-h)/2,setsar=1,format=yuv420p[base];"
        "[base]split=2[teaser_src][main_src];"
        f"[teaser_src]trim=start={teaser_local_start:.3f}:duration={teaser_duration:.3f},setpts=PTS-STARTPTS[teaser];"
        f"[main_src]trim=start=0:duration={main_duration:.3f},setpts=PTS-STARTPTS[main];"
        "[teaser][main]concat=n=2:v=1:a=0[raw]"
    )
    if HAS_DRAWTEXT_FILTER:
        hook_filter_path = _escape_filter_path(hook_path)
        base_graph += (
            f";[raw]drawtext=fontfile='{FONT_PATH}':textfile='{hook_filter_path}':"
            "fontcolor=white:fontsize=66:borderw=4:bordercolor=black:"
            "box=1:boxcolor=black@0.50:boxborderw=28:"
            "x=(w-text_w)/2:y=235:line_spacing=12:"
            f"enable='between(t,0,{teaser_duration:.3f})'[hooked]"
        )
        current_video_label = "hooked"
    else:
        base_graph += ";[raw]null[hooked]"
        current_video_label = "hooked"
        print("drawtext unavailable: teaser is included but hook text will remain in the editorial sidecar.")

    if caption_word_count and HAS_SUBTITLES_FILTER:
        ass_filter_path = _escape_filter_path(ass_path)
        base_graph += f";[{current_video_label}]subtitles='{ass_filter_path}'[v]"
        print(f"Burning in {caption_word_count} caption words and a {teaser_duration:.1f}s opening teaser.")
    else:
        base_graph += f";[{current_video_label}]null[v]"
        if not caption_word_count:
            print(f"No speech detected for candidate {idx}; exporting without captions rather than inventing dialogue.")
        else:
            print("FFmpeg subtitles filter unavailable; subtitle sidecars will still be included.")

    if HAS_AUDIO:
        audio_graph = (
            f";[0:a]asplit=2[teaser_audio_src][main_audio_src];"
            f"[teaser_audio_src]atrim=start={teaser_local_start:.3f}:duration={teaser_duration:.3f},asetpts=PTS-STARTPTS[teaser_audio];"
            f"[main_audio_src]atrim=start=0:duration={main_duration:.3f},asetpts=PTS-STARTPTS[main_audio];"
            "[teaser_audio][main_audio]concat=n=2:v=0:a=1[a]"
        )
        base_graph += audio_graph

    cmd = ["ffmpeg", "-y", "-ss", f"{start:.3f}", "-i", VIDEO_PATH, "-t", f"{duration:.3f}",
           "-filter_complex", base_graph, "-map", "[v]"]
    if HAS_AUDIO:
        cmd += ["-map", "[a]", "-c:a", "aac", "-b:a", "160k"]
    cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "22", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(out_path)]
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Rendering {out_path.name} failed. FFmpeg said:\n{result.stderr[-4500:]}")
    print(f"Created {out_path.name} ({duration:.1f}s) | category: {pack['category']} | caption words: {caption_word_count}")

print("Growth-ready candidate rendering complete.")


In [ ]:
# 9) Preview, create the cross-platform publishing pack, and download the ZIP.
from IPython.display import Video, display
import zipfile, csv, json

for path in sorted(OUTPUT_DIR.glob("candidate_*.mp4")):
    print(path.name)
    display(Video(str(path), embed=True, width=360))

PUBLISHING_PACK_PATH = WORK_DIR / "publishing_pack.txt"
pack_lines = [
    f"Creator: {CREATOR_NAME}", f"Game: {GAME_NAME or 'Not specified'}", f"Source: {SOURCE_URL}",
    f"Session mode: {SESSION_MODE}", f"Rights basis supplied: {RIGHTS_BASIS}",
    "", "IMPORTANT: These are editing and metadata suggestions, not proof of rights or monetization eligibility.",
    "Review gameplay context, accuracy, creator permission, music rights, captions, and titles before publishing.",
    ""
]
for pack in EDITORIAL_PACK:
    pack_lines += [
        f"CANDIDATE {pack['candidate']} — {pack['category']} ({pack['category_confidence']} confidence)",
        f"Hook overlay: {pack['hook_text']}", "Title options:",
        *[f"  {n}. {title}" for n, title in enumerate(pack['title_options'], 1)],
        "Description:", pack['description'], "Hashtags: " + " ".join(pack['hashtags']),
        "Pinned question: " + pack['pinned_question'], "TikTok caption:", pack['tiktok_caption'],
        "Instagram caption:", pack['instagram_caption'], "Original commentary guidance:", pack['voiceover_guidance'],
        "Review note: " + pack['editorial_review_note'], ""
    ]
PUBLISHING_PACK_PATH.write_text("\n".join(pack_lines), encoding="utf-8")

# Starter tracker: add one row per candidate. Fill in actual Studio metrics after publishing.
TRACKER_PATH = WORK_DIR / "performance_tracker_template.csv"
tracker_fields = [
    "Creator", "Game", "Candidate", "Category", "Title used", "Publish date",
    "Views 24h", "Engaged views 24h", "Stayed to watch % 24h", "Average view duration 24h",
    "Average percentage viewed 24h", "Views 7d", "Engaged views 7d", "Stayed to watch % 7d",
    "Average view duration 7d", "Average percentage viewed 7d", "Likes", "Comments", "Shares",
    "Subscribers gained", "Views 28d", "Learning note", "Next test"
]
with TRACKER_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=tracker_fields)
    writer.writeheader()
    for pack in EDITORIAL_PACK:
        writer.writerow({
            "Creator": CREATOR_NAME, "Game": GAME_NAME, "Candidate": pack['candidate'],
            "Category": pack['category'], "Title used": pack['title_options'][0]
        })

ZIP_PATH = WORK_DIR / f"{CREATOR_SLUG}_growth_ready_candidates.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for pattern in ("candidate_*.mp4", "candidate_*.ass", "candidate_*.srt", "candidate_*_editorial.txt", "candidate_*_hook.txt"):
        for path in sorted(OUTPUT_DIR.glob(pattern)):
            zf.write(path, arcname=path.name)
    zf.write(PUBLISHING_PACK_PATH, arcname="publishing_pack.txt")
    zf.write(TRACKER_PATH, arcname="performance_tracker_template.csv")
    with zf.open("candidate_notes.txt", "w") as f:
        notes = [
            f"Creator: {CREATOR_NAME}", f"Game: {GAME_NAME}", f"Source: {SOURCE_URL}",
            f"Session mode: {SESSION_MODE}", f"Permission/licence basis supplied: {RIGHTS_BASIS}",
            "Captions are Whisper-generated estimates. Check names, slang, timing, and context.",
            "The opening teaser replays a high-energy 1.6-second section before the main sequence.",
            "Hook/category/title suggestions are audio/transcript heuristics, not visual scene analysis.",
            "Test mode is private testing only and does not grant reposting rights.", ""
        ]
        for i, item in enumerate(selected, 1):
            notes.extend([f"Candidate {i}: {item['start']:.2f}s–{item['end']:.2f}s | score {item['score']:.3f}", item['transcript'], ""])
        f.write("\n".join(notes).encode("utf-8"))

print("Included: captioned MP4s, SRT/ASS subtitles, per-candidate editorial notes, publishing copy, and a performance-tracker CSV.")
from google.colab import files
files.download(str(ZIP_PATH))


## Output and publishing workflow

Each ZIP contains:
- `candidate_XX.mp4` — vertical video, brief action teaser, opening hook, and progressive captions when speech is detected.
- `candidate_XX.srt` and `.ass` — subtitle sidecars.
- `candidate_XX_editorial.txt` — category, hook, title options, description, hashtags, pinned question, and cross-platform copy.
- `publishing_pack.txt` — consolidated YouTube/TikTok/Instagram copy suggestions.
- `performance_tracker_template.csv` — columns for 24-hour, 7-day, and 28-day performance notes.

The teaser is a short replay from the strongest audio-activity window, then the selected clip plays. It is an automated heuristic; inspect that it previews the right moment and does not mislead viewers. The opening hook uses nearby recognised speech when available, otherwise a neutral category-level prompt. Categories are tentative and based on transcript keywords. Visual motion is only a ranking aid; it does not understand the scene or identify specific gameplay events.

The tool does not automatically upload, post comments, share to other platforms, or add synthetic voiceover. For meaningful editorial value, add genuine original context/commentary where appropriate. Verify creator permissions, third-party music rights, platform rules, and all outputs before publishing. Captions, reframing, hooks, or permission alone do not guarantee YouTube monetization approval.

If Kick extraction fails, use only an authorised source file as fallback. Do not use cookies or logged-in credentials to bypass subscriber-only content or access controls.
